In [1]:
# PROCESS 1 – SETUP

from pathlib import Path
import sys

import numpy as np
import pandas as pd

APP_ROOT = Path(
    "/Users/selvianiaulyah/Documents/AI-Agent/gallup-engagement-agent"
)

DATA_PATH = (
    APP_ROOT
    / "data"
    / "dev"
    / "gallup_v2_5K"
    / "agent_data"
)

REFERENCE_CODE = Path(
    "/Users/selvianiaulyah/Documents/AI-Agent/"
    "Business-Practicum-Group-10-/synthetic_data_v2/gallup_v2_code"
)

sys.path.insert(0, str(REFERENCE_CODE))

import stats_tools as st

print("Data path:", DATA_PATH)
print("Reference code:", REFERENCE_CODE)
print("Q12 items:", st.ITEMS)

Data path: /Users/selvianiaulyah/Documents/AI-Agent/gallup-engagement-agent/data/dev/gallup_v2_5K/agent_data
Reference code: /Users/selvianiaulyah/Documents/AI-Agent/Business-Practicum-Group-10-/synthetic_data_v2/gallup_v2_code
Q12 items: ['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12']


In [2]:
# PROCESS 2 – LOAD AGENT-VISIBLE DATA

survey = pd.read_csv(DATA_PATH / "survey_responses.csv")
indiv = pd.read_csv(DATA_PATH / "individual_outcomes.csv")
team = pd.read_csv(DATA_PATH / "team_outcomes.csv")

print("Survey:", survey.shape)
print("Individual outcomes:", indiv.shape)
print("Team outcomes:", team.shape)

Survey: (10000, 22)
Individual outcomes: (10000, 9)
Team outcomes: (800, 4)


In [3]:
# PROCESS 3 – BUILD ANALYTICAL SAMPLE

df = survey.merge(
    indiv,
    on=["wave", "employee_id"]
)

w1_all = df[df["wave"] == 1].copy()

w1 = w1_all[
    w1_all["responded"] == 1
].copy()

print("Merged rows:", len(df))
print("Wave 1 rows:", len(w1_all))
print("Wave 1 respondents:", len(w1))

print()
print("Response rate:")
print(f"{w1_all['responded'].mean():.1%}")

Merged rows: 10000
Wave 1 rows: 5000
Wave 1 respondents: 4024

Response rate:
80.5%


In [4]:
# PROCESS 4 – RUN STANDARD MODELS

turnover_fit = st.turnover_model(w1)

absence_fit = st.absence_model(w1)

performance_fit = st.performance_model(w1)

print("Turnover n:", turnover_fit.n)
print("Absence n:", absence_fit.n)
print("Performance n:", performance_fit.n)

Turnover n: 4024
Absence n: 4024
Performance n: 4024


In [5]:
# PROCESS 5 – TURNOVER RESULTS

turnover_rows = []

for item in st.ITEMS:
    odds_ratio, ci_low, ci_high = turnover_fit.ratio(item)

    turnover_rows.append({
        "item": item,
        "outcome": "turnover",
        "effect_type": "Odds Ratio",
        "effect": odds_ratio,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": turnover_fit.pvalues[item],
        "n": turnover_fit.n,
    })

turnover_results = pd.DataFrame(turnover_rows)

turnover_results


,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n
0,Q01,turnover,Odds Ratio,1.092183,0.963626,1.237890,1.675663e-01,4024
1,Q02,turnover,Odds Ratio,0.977927,0.864345,1.106435,7.230925e-01,4024
2,Q03,turnover,Odds Ratio,0.974999,0.858794,1.106927,6.957715e-01,4024
3,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024
4,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024
5,Q06,turnover,Odds Ratio,1.058798,0.945430,1.185760,3.227598e-01,4024
6,Q07,turnover,Odds Ratio,0.946026,0.857558,1.043621,2.680231e-01,4024
7,Q08,turnover,Odds Ratio,0.983479,0.864871,1.118352,7.994465e-01,4024
8,Q09,turnover,Odds Ratio,1.124556,0.984840,1.284093,8.286797e-02,4024
9,Q10,turnover,Odds Ratio,1.045057,0.933236,1.170276,4.453036e-01,4024


In [6]:
# PROCESS 6 – ABSENCE RESULTS

absence_rows = []

for item in st.ITEMS:
    rate_ratio, ci_low, ci_high = absence_fit.ratio(item)

    absence_rows.append({
        "item": item,
        "outcome": "absence",
        "effect_type": "Incidence Rate Ratio",
        "effect": rate_ratio,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": absence_fit.pvalues[item],
        "n": absence_fit.n,
    })

absence_results = pd.DataFrame(absence_rows)

absence_results

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n
0,Q01,absence,Incidence Rate Ratio,1.016308,0.963258,1.072279,5.542634e-01,4024
1,Q02,absence,Incidence Rate Ratio,1.013232,0.976236,1.051631,4.885273e-01,4024
2,Q03,absence,Incidence Rate Ratio,1.034521,0.993270,1.077486,1.021129e-01,4024
3,Q04,absence,Incidence Rate Ratio,1.037561,1.000091,1.076435,4.943561e-02,4024
4,Q05,absence,Incidence Rate Ratio,0.876323,0.842623,0.911371,4.155788e-11,4024
5,Q06,absence,Incidence Rate Ratio,0.983652,0.950128,1.018359,3.515004e-01,4024
6,Q07,absence,Incidence Rate Ratio,0.812396,0.782785,0.843128,5.497937e-28,4024
7,Q08,absence,Incidence Rate Ratio,1.009623,0.963375,1.058092,6.889145e-01,4024
8,Q09,absence,Incidence Rate Ratio,1.022905,0.980669,1.066961,2.925076e-01,4024
9,Q10,absence,Incidence Rate Ratio,0.994203,0.959447,1.030218,7.488050e-01,4024


In [7]:
# PROCESS 7 – PERFORMANCE RESULTS

performance_rows = []

for item in st.ITEMS:
    ci_low, ci_high = performance_fit.ci(item)

    performance_rows.append({
        "item": item,
        "outcome": "performance",
        "effect_type": "Standardized Beta",
        "effect": performance_fit.params[item],
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": performance_fit.pvalues[item],
        "n": performance_fit.n,
    })

performance_results = pd.DataFrame(performance_rows)

performance_results

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n
0,Q01,performance,Standardized Beta,0.126114,0.095886,0.156341,2.904867e-16,4024
1,Q02,performance,Standardized Beta,-0.018285,-0.047687,0.011117,2.228901e-01,4024
2,Q03,performance,Standardized Beta,0.001324,-0.030294,0.032941,9.346072e-01,4024
3,Q04,performance,Standardized Beta,-0.006269,-0.036278,0.023739,6.821990e-01,4024
4,Q05,performance,Standardized Beta,0.004455,-0.024665,0.033575,7.642974e-01,4024
5,Q06,performance,Standardized Beta,-0.005380,-0.035950,0.025189,7.301218e-01,4024
6,Q07,performance,Standardized Beta,0.096653,0.070629,0.122678,3.356378e-13,4024
7,Q08,performance,Standardized Beta,0.005741,-0.023649,0.035132,7.018170e-01,4024
8,Q09,performance,Standardized Beta,-0.000209,-0.029519,0.029101,9.888604e-01,4024
9,Q10,performance,Standardized Beta,-0.012246,-0.039462,0.014969,3.777992e-01,4024


In [8]:
# PROCESS 8 – COMBINE ALL 36 TESTS

results = pd.concat(
    [
        turnover_results,
        absence_results,
        performance_results,
    ],
    ignore_index=True
)

print("Number of tests:", len(results))

results

Number of tests: 36


,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n
0,Q01,turnover,Odds Ratio,1.092183,0.963626,1.237890,1.675663e-01,4024
1,Q02,turnover,Odds Ratio,0.977927,0.864345,1.106435,7.230925e-01,4024
2,Q03,turnover,Odds Ratio,0.974999,0.858794,1.106927,6.957715e-01,4024
3,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024
4,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024
5,Q06,turnover,Odds Ratio,1.058798,0.945430,1.185760,3.227598e-01,4024
6,Q07,turnover,Odds Ratio,0.946026,0.857558,1.043621,2.680231e-01,4024
7,Q08,turnover,Odds Ratio,0.983479,0.864871,1.118352,7.994465e-01,4024
8,Q09,turnover,Odds Ratio,1.124556,0.984840,1.284093,8.286797e-02,4024
9,Q10,turnover,Odds Ratio,1.045057,0.933236,1.170276,4.453036e-01,4024


In [9]:
# PROCESS 9 – BENJAMINI-HOCHBERG FDR

results["fdr_significant"] = st.bh_fdr(
    results["p_value"].to_numpy(),
    q=0.05
)

results

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n,fdr_significant
0,Q01,turnover,Odds Ratio,1.092183,0.963626,1.237890,1.675663e-01,4024,False
1,Q02,turnover,Odds Ratio,0.977927,0.864345,1.106435,7.230925e-01,4024,False
2,Q03,turnover,Odds Ratio,0.974999,0.858794,1.106927,6.957715e-01,4024,False
3,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024,True
4,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024,True
5,Q06,turnover,Odds Ratio,1.058798,0.945430,1.185760,3.227598e-01,4024,False
6,Q07,turnover,Odds Ratio,0.946026,0.857558,1.043621,2.680231e-01,4024,False
7,Q08,turnover,Odds Ratio,0.983479,0.864871,1.118352,7.994465e-01,4024,False
8,Q09,turnover,Odds Ratio,1.124556,0.984840,1.284093,8.286797e-02,4024,False
9,Q10,turnover,Odds Ratio,1.045057,0.933236,1.170276,4.453036e-01,4024,False


In [10]:
# PROCESS 10 – MEANINGFUL EFFECT FILTER

def is_meaningful(row):
    if row["outcome"] in ["turnover", "absence"]:
        return (
            row["effect"] < 0.91
            or row["effect"] > 1.10
        )

    if row["outcome"] == "performance":
        return abs(row["effect"]) >= 0.05

    return False


results["meaningful_effect"] = results.apply(
    is_meaningful,
    axis=1
)

results["reportable"] = (
    results["fdr_significant"]
    & results["meaningful_effect"]
)

results

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n,fdr_significant,meaningful_effect,reportable
0,Q01,turnover,Odds Ratio,1.092183,0.963626,1.237890,1.675663e-01,4024,False,False,False
1,Q02,turnover,Odds Ratio,0.977927,0.864345,1.106435,7.230925e-01,4024,False,False,False
2,Q03,turnover,Odds Ratio,0.974999,0.858794,1.106927,6.957715e-01,4024,False,False,False
3,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024,True,True,True
4,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024,True,True,True
5,Q06,turnover,Odds Ratio,1.058798,0.945430,1.185760,3.227598e-01,4024,False,False,False
6,Q07,turnover,Odds Ratio,0.946026,0.857558,1.043621,2.680231e-01,4024,False,False,False
7,Q08,turnover,Odds Ratio,0.983479,0.864871,1.118352,7.994465e-01,4024,False,False,False
8,Q09,turnover,Odds Ratio,1.124556,0.984840,1.284093,8.286797e-02,4024,False,True,False
9,Q10,turnover,Odds Ratio,1.045057,0.933236,1.170276,4.453036e-01,4024,False,False,False


In [11]:
# PROCESS 11 – REPORTABLE FINDINGS

reportable = (
    results[
        results["reportable"]
    ]
    .sort_values(
        ["outcome", "effect"]
    )
    .reset_index(drop=True)
)

reportable

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n,fdr_significant,meaningful_effect,reportable
0,Q07,absence,Incidence Rate Ratio,0.812396,0.782785,0.843128,5.497937e-28,4024,True,True,True
1,Q05,absence,Incidence Rate Ratio,0.876323,0.842623,0.911371,4.155788e-11,4024,True,True,True
2,Q11,performance,Standardized Beta,0.075308,0.045763,0.104853,5.857934e-07,4024,True,True,True
3,Q07,performance,Standardized Beta,0.096653,0.070629,0.122678,3.356378e-13,4024,True,True,True
4,Q01,performance,Standardized Beta,0.126114,0.095886,0.156341,2.904867e-16,4024,True,True,True
5,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024,True,True,True
6,Q12,turnover,Odds Ratio,0.839604,0.751837,0.937616,1.912812e-03,4024,True,True,True
7,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024,True,True,True


In [12]:
# PROCESS 12 – X2 PRIOR-PERFORMANCE SENSITIVITY

z = pd.DataFrame(
    {
        c: st.zscore(w1[c])
        for c in ["performance_rating", "Q11"]
    },
    index=w1.index
)

q11_raw_fit = st.ols(
    z["performance_rating"],
    st.design(z, ["Q11"])
)

q11_raw_beta = q11_raw_fit.params["Q11"]
q11_controlled_beta = performance_fit.params["Q11"]

q11_reduction = (
    1 - abs(q11_controlled_beta / q11_raw_beta)
)

print("Q11 raw beta:", round(q11_raw_beta, 4))
print("Q11 controlled beta:", round(q11_controlled_beta, 4))
print("Reduction:", f"{q11_reduction:.1%}")

Q11 raw beta: 0.2739
Q11 controlled beta: 0.0753
Reduction: 72.5%


In [13]:
# PROCESS 13 – X3 TEAM-LEVEL CONFOUNDING CHECK

a = w1.assign(
    y=np.log1p(w1["unplanned_absence_days_6m"]),
    q01_low=(w1["Q01"] < 3).astype(float),
    night=(w1["shift_type"] == "Night").astype(float),
)

xs = list(st.ITEMS) + [
    "q01_low",
    "tenure_years",
    "night",
]

department_fit = st.ols(
    a["y"],
    st.design(a, xs, ["department"]),
    a["team_id"],
)

within_team = st.demean(
    a,
    ["y"] + xs,
    "team_id",
)

team_fit = st.ols(
    within_team["y"],
    st.design(within_team, xs),
    a["team_id"],
)

q07_department = department_fit.params["Q07"]
q07_within_team = team_fit.params["Q07"]

shrinkage = 1 - abs(
    q07_within_team / q07_department
)

print(
    "Q07 department-controlled:",
    round(q07_department, 4)
)

print(
    "Q07 within-team:",
    round(q07_within_team, 4)
)

print(
    "Shrinkage:",
    f"{shrinkage:.1%}"
)

Q07 department-controlled: -0.1432
Q07 within-team: -0.0088
Shrinkage: 93.8%


In [14]:
# PROCESS 14 – TEAM CLUSTERING CHECK

icc, icc_p = st.icc_oneway(
    a["y"],
    a["team_id"]
)

print("Absence team ICC:", round(icc, 4))
print("p-value:", icc_p)

Absence team ICC: 0.1041
p-value: 5.286915390010921e-31


In [15]:
# PROCESS 15 – SIMPSON'S PARADOX CHECK

z_perf = pd.DataFrame(
    {
        c: st.zscore(w1[c])
        for c in ["performance_rating", "Q07"]
    },
    index=w1.index,
)

z_perf["department"] = w1["department"]

q07_pooled_fit = st.ols(
    z_perf["performance_rating"],
    st.design(z_perf, ["Q07"]),
)

q07_adjusted_fit = st.ols(
    z_perf["performance_rating"],
    st.design(
        z_perf,
        ["Q07"],
        ["department"],
    ),
)

q07_pooled = q07_pooled_fit.params["Q07"]
q07_adjusted = q07_adjusted_fit.params["Q07"]

print("Q07 pooled beta:", round(q07_pooled, 4))
print(
    "Q07 department-adjusted beta:",
    round(q07_adjusted, 4)
)

print(
    "Direction flip:",
    np.sign(q07_pooled) != np.sign(q07_adjusted)
)

Q07 pooled beta: -0.0848
Q07 department-adjusted beta: 0.1753
Direction flip: True


In [16]:
# PROCESS 16 – ROBUSTNESS CLASSIFICATION

robust_results = results.copy()

robust_results["robustness_status"] = np.where(
    robust_results["reportable"],
    "ROBUST",
    "NOT_REPORTABLE"
)

robust_results["priority_eligible"] = robust_results["reportable"]

robust_results["diagnostic_code"] = ""
robust_results["diagnostic_reason"] = ""

In [17]:
# Q11 -> Performance : X2
mask = (
    (robust_results["item"] == "Q11")
    & (robust_results["outcome"] == "performance")
)

robust_results.loc[mask, "robustness_status"] = "CAUTION"
robust_results.loc[mask, "priority_eligible"] = False
robust_results.loc[mask, "diagnostic_code"] = "X2_PRIOR_OUTCOME"
robust_results.loc[
    mask,
    "diagnostic_reason"
] = (
    f"Effect decreased by {q11_reduction:.1%} "
    "after controlling for prior performance."
)

In [18]:
# Q07 -> Absence : X3
mask = (
    (robust_results["item"] == "Q07")
    & (robust_results["outcome"] == "absence")
)

robust_results.loc[mask, "robustness_status"] = "CAUTION"
robust_results.loc[mask, "priority_eligible"] = False
robust_results.loc[mask, "diagnostic_code"] = "X3_TEAM_CONFOUNDING"
robust_results.loc[
    mask,
    "diagnostic_reason"
] = (
    f"Effect shrank by {shrinkage:.1%} within teams; "
    f"team ICC={icc:.3f}."
)

In [19]:
# Q07 -> Performance : X6
mask = (
    (robust_results["item"] == "Q07")
    & (robust_results["outcome"] == "performance")
)

robust_results.loc[mask, "robustness_status"] = "CAUTION"
robust_results.loc[mask, "priority_eligible"] = False
robust_results.loc[mask, "diagnostic_code"] = "X6_SIMPSON"
robust_results.loc[
    mask,
    "diagnostic_reason"
] = (
    f"Direction flips from pooled {q07_pooled:.3f} "
    f"to department-adjusted {q07_adjusted:.3f}."
)

In [20]:
# PROCESS 17 – FINAL ANALYTICAL FINDINGS

final_findings = (
    robust_results[
        robust_results["reportable"]
    ][
        [
            "item",
            "outcome",
            "effect_type",
            "effect",
            "ci_low",
            "ci_high",
            "p_value",
            "n",
            "fdr_significant",
            "meaningful_effect",
            "robustness_status",
            "priority_eligible",
            "diagnostic_code",
            "diagnostic_reason",
        ]
    ]
    .sort_values(
        ["robustness_status", "outcome", "item"]
    )
    .reset_index(drop=True)
)

final_findings

,item,outcome,effect_type,effect,ci_low,ci_high,p_value,n,fdr_significant,meaningful_effect,robustness_status,priority_eligible,diagnostic_code,diagnostic_reason
0,Q07,absence,Incidence Rate Ratio,0.812396,0.782785,0.843128,5.497937e-28,4024,True,True,CAUTION,False,X3_TEAM_CONFOUNDING,Effect shrank by 93.8% within teams; team ICC=...
1,Q07,performance,Standardized Beta,0.096653,0.070629,0.122678,3.356378e-13,4024,True,True,CAUTION,False,X6_SIMPSON,Direction flips from pooled -0.085 to departme...
2,Q11,performance,Standardized Beta,0.075308,0.045763,0.104853,5.857934e-07,4024,True,True,CAUTION,False,X2_PRIOR_OUTCOME,Effect decreased by 72.5% after controlling fo...
3,Q05,absence,Incidence Rate Ratio,0.876323,0.842623,0.911371,4.155788e-11,4024,True,True,ROBUST,True,,
4,Q01,performance,Standardized Beta,0.126114,0.095886,0.156341,2.904867e-16,4024,True,True,ROBUST,True,,
5,Q04,turnover,Odds Ratio,0.691280,0.614269,0.777946,8.970176e-10,4024,True,True,ROBUST,True,,
6,Q05,turnover,Odds Ratio,0.841311,0.751146,0.942299,2.812530e-03,4024,True,True,ROBUST,True,,
7,Q12,turnover,Odds Ratio,0.839604,0.751837,0.937616,1.912812e-03,4024,True,True,ROBUST,True,,


In [21]:
# PROCESS 18 – ROBUSTNESS SUMMARY

print(
    final_findings["robustness_status"]
    .value_counts()
)

print()

print(
    "Priority-eligible findings:",
    final_findings["priority_eligible"].sum()
)

robustness_status
ROBUST     5
CAUTION    3
Name: count, dtype: int64

Priority-eligible findings: 5


In [22]:
# PROCESS 19 – LOAD HEALTH REPORT

import json

HEALTH_PATH = (
    APP_ROOT
    / "data"
    / "dev"
    / "gallup_v2_5K"
    / "reports"
    / "health_report.json"
)

with open(HEALTH_PATH, "r") as f:
    health_report = json.load(f)

health_summary = health_report["summary"]
health_checks = health_report["checks"]

print("Checks:", health_summary["checks_run"])
print("PASS:", health_summary["pass"])
print("WARN:", health_summary["warn"])
print("FAIL:", health_summary["fail"])
print()
print("Analysis allowed:")
print(health_summary["analysis_allowed"])

Checks: 45
PASS: 32
WARN: 13
FAIL: 0

Analysis allowed:
{'turnover': True, 'absence': True, 'performance': True}


In [23]:
# PROCESS 20 – CAVEAT CONFIGURATION AND SELECTION

CAVEAT_MESSAGES = {
    "NR_LEAVERS": (
        "Survey response rates were lower among employees who later left "
        "(57.7%) than among stayers (85.6%). This nonresponse pattern may "
        "cause the observed turnover association to be understated."
    ),

    "NR_SMD_unplanned_absence_days_6m": (
    "Employees who responded to the survey differed in absenteeism "
    "from those who did not respond, so nonresponse may affect the "
    "estimated association."
    ),

    "MS_DIF": (
        "The item shows different response behavior across departments, "
        "so department context should be considered when interpreting the result."
    ),

    "RS_EXTREME": (
        "About 7.8% of respondents showed extreme response patterns. "
        "This is considered a weak data-quality caution."
    ),

    "NR_GAP_department": (
        "Survey response rates differed across departments, with a "
        "6.5 percentage-point gap between the lowest and highest response rates."
    ),

    "NR_GAP_pay_band": (
        "Survey response rates differed across pay bands, with a "
        "5.2 percentage-point gap between the lowest and highest response rates."
    ),
}


PRIMARY_CAVEATS = {
    "turnover": {
        "NR_LEAVERS",
    },

    "absence": {
        "NR_SMD_unplanned_absence_days_6m",
        "MS_DIF",
    },

    "performance": {
        "MS_DIF",
    },
}


def select_caveats(
    outcome,
    question_scope="overall",
    item=None
):
    selected = []

    primary_codes = PRIMARY_CAVEATS.get(
        outcome,
        set()
    )

    for check in health_checks:

        if check["status"] != "WARN":
            continue

        affects = check.get(
            "affects",
            []
        )

        # Warning tidak relevan dengan outcome
        if (
            "all" not in affects
            and outcome not in affects
        ):
            continue

        code = check["code"]

        # Small-team warning hanya untuk team/group analysis
        if (
            code == "COV_SMALL_TEAMS"
            and question_scope not in ["team", "group"]
        ):
            continue

        # Survivorship hanya untuk trend analysis
        if (
            code.startswith("SV_")
            and question_scope != "trend"
        ):
            continue

        # Sudah ditangani oleh fixed statistical model
        if code in {
            "OL_PERF_GAP_department",
            "OL_TURNOVER_GAP_department",
            "OL_PRIOR_ANCHOR",
        }:
            continue

        # MS_DIF hanya berlaku untuk item yang memang terdeteksi
        if code == "MS_DIF":

            affected_items = (
                check
                .get("metrics", {})
                .get("items", [])
            )

            if (
                item is not None
                and item not in affected_items
            ):
                continue

        # Simpson hanya berlaku pada item/outcome pair tertentu
        if code == "AG_SIMPSON":

            pairs = (
                check
                .get("metrics", {})
                .get("pairs", [])
            )

            matching_pair = any(
                p.get("item") == item
                and p.get("outcome") == outcome
                for p in pairs
            )

            if (
                item is not None
                and not matching_pair
            ):
                continue

        # Overall question hanya menampilkan material caveats
        if (
            question_scope == "overall"
            and code not in primary_codes
        ):
            continue

        selected.append({
            "code": code,

            "detail": CAVEAT_MESSAGES.get(
                code,
                check["detail"]
            ),

            "evidence_strength":
                check.get(
                    "evidence_strength"
                ),

            "actions":
                check.get(
                    "actions",
                    []
                ),
        })

    return selected

In [24]:
# PROCESS 21 – MODEL METADATA

MODEL_METADATA = {
    "turnover": {
        "method": "Logistic Regression",
        "effect_type": "Odds Ratio",
        "controls": [
            "tenure_years",
            "department",
            "role_level",
            "pay_band",
        ],
        "cluster": "team_id",
    },

    "absence": {
        "method": "Negative Binomial Regression",
        "effect_type": "Incidence Rate Ratio",
        "controls": [
            "tenure_years",
            "department",
            "shift_type",
            "Q01_below_3_threshold",
        ],
        "cluster": "team_id",
    },

    "performance": {
        "method": "Standardized Linear Regression",
        "effect_type": "Standardized Beta",
        "controls": [
            "tenure_years",
            "department",
            "prior_performance_rating",
        ],
        "cluster": "team_id",
    },
}

In [25]:
# PROCESS 22 – CALCULATION ID

import hashlib


def make_calculation_id(
    dataset,
    outcome,
    item,
    method
):
    raw = (
        f"{dataset}|"
        f"{outcome}|"
        f"{item}|"
        f"{method}"
    )

    short_hash = hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()[:10].upper()

    return (
        f"{outcome.upper()}-"
        f"{item}-"
        f"{short_hash}"
    )

In [26]:
make_calculation_id(
    "gallup_v2_5K",
    "turnover",
    "Q04",
    "Logistic Regression"
)

'TURNOVER-Q04-BF5A62B9B6'

In [27]:
# PROCESS 23 – BUILD STRUCTURED FINDING

def build_finding_payload(
    row,
    question_scope="overall"
):
    outcome = row["outcome"]

    meta = MODEL_METADATA[outcome]

    payload = {
        "dataset": "gallup_v2_5K",

        "item": row["item"],
        "outcome": outcome,

        "method":
            meta["method"],

        "effect_type":
            row["effect_type"],

        "effect": round(
            float(row["effect"]),
            4
        ),

        "confidence_interval": {
            "low": round(
                float(row["ci_low"]),
                4
            ),

            "high": round(
                float(row["ci_high"]),
                4
            ),

            "level": 0.95,
        },

        "p_value": float(
            row["p_value"]
        ),

        "fdr_significant": bool(
            row["fdr_significant"]
        ),

        "meaningful_effect": bool(
            row["meaningful_effect"]
        ),

        "sample_size": int(
            row["n"]
        ),

        "robustness": {
            "status":
                row["robustness_status"],

            "priority_eligible": bool(
                row["priority_eligible"]
            ),

            "diagnostic_code": (
                row["diagnostic_code"]
                if row["diagnostic_code"]
                else None
            ),

            "diagnostic_reason": (
                row["diagnostic_reason"]
                if row["diagnostic_reason"]
                else None
            ),
        },

        "controls":
            meta["controls"],

        "clustered_by":
            meta["cluster"],

        "health": {
            "analysis_allowed": bool(
                health_summary[
                    "analysis_allowed"
                ][outcome]
            ),

            "caveats": select_caveats(
                outcome=outcome,
                question_scope=question_scope,
                item=row["item"]
            ),
        },
    }

    payload["calculation_id"] = (
        make_calculation_id(
            payload["dataset"],
            outcome,
            row["item"],
            meta["method"],
        )
    )

    return payload

In [28]:
# PROCESS 24 – TEST STRUCTURED PAYLOAD

q04_turnover_row = final_findings[
    (final_findings["item"] == "Q04")
    & (final_findings["outcome"] == "turnover")
].iloc[0]

q04_payload = build_finding_payload(
    q04_turnover_row
)

print(
    json.dumps(
        q04_payload,
        indent=2
    )
)

{
  "dataset": "gallup_v2_5K",
  "item": "Q04",
  "outcome": "turnover",
  "method": "Logistic Regression",
  "effect_type": "Odds Ratio",
  "effect": 0.6913,
  "confidence_interval": {
    "low": 0.6143,
    "high": 0.7779,
    "level": 0.95
  },
  "p_value": 8.970176340330781e-10,
  "fdr_significant": true,
  "meaningful_effect": true,
  "sample_size": 4024,
  "robustness": {
    "status": "ROBUST",
    "priority_eligible": true,
    "diagnostic_code": null,
    "diagnostic_reason": null
  },
  "controls": [
    "tenure_years",
    "department",
    "role_level",
    "pay_band"
  ],
  "clustered_by": "team_id",
  "health": {
    "analysis_allowed": true,
    "caveats": [
      {
        "code": "NR_LEAVERS",
        "detail": "Survey response rates were lower among employees who later left (57.7%) than among stayers (85.6%). This nonresponse pattern may cause the observed turnover association to be understated.",
        "evidence_strength": "strong",
        "actions": [
          "

In [29]:
# PROCESS 25 – BUILD ALL STRUCTURED FINDINGS

structured_findings = [
    build_finding_payload(row)
    for _, row in final_findings.iterrows()
]

print(
    json.dumps(
        structured_findings,
        indent=2
    )
)

[
  {
    "dataset": "gallup_v2_5K",
    "item": "Q07",
    "outcome": "absence",
    "method": "Negative Binomial Regression",
    "effect_type": "Incidence Rate Ratio",
    "effect": 0.8124,
    "confidence_interval": {
      "low": 0.7828,
      "high": 0.8431,
      "level": 0.95
    },
    "p_value": 5.497937113130814e-28,
    "fdr_significant": true,
    "meaningful_effect": true,
    "sample_size": 4024,
    "robustness": {
      "status": "CAUTION",
      "priority_eligible": false,
      "diagnostic_code": "X3_TEAM_CONFOUNDING",
      "diagnostic_reason": "Effect shrank by 93.8% within teams; team ICC=0.104."
    },
    "controls": [
      "tenure_years",
      "department",
      "shift_type",
      "Q01_below_3_threshold"
    ],
    "clustered_by": "team_id",
    "health": {
      "analysis_allowed": true,
      "caveats": [
        {
          "code": "NR_SMD_unplanned_absence_days_6m",
          "detail": "Employees who responded to the survey differed in absenteeism from 

In [30]:
# PROCESS 26 – DETERMINISTIC BUSINESS METRICS

def add_business_metrics(payload):
    payload = payload.copy()

    effect = payload["effect"]
    effect_type = payload["effect_type"]

    if effect_type == "Odds Ratio":
        payload["business_metrics"] = {
            "relative_odds_change_pct": round(
                (effect - 1) * 100,
                1
            ),
            "interpretation_unit":
                "odds, not probability"
        }

    elif effect_type == "Incidence Rate Ratio":
        payload["business_metrics"] = {
            "relative_rate_change_pct": round(
                (effect - 1) * 100,
                1
            )
        }

    elif effect_type == "Standardized Beta":
        payload["business_metrics"] = {
            "standardized_beta": effect
        }

    return payload


q04_llm_payload = add_business_metrics(
    q04_payload
)

print(
    json.dumps(
        q04_llm_payload,
        indent=2
    )
)

{
  "dataset": "gallup_v2_5K",
  "item": "Q04",
  "outcome": "turnover",
  "method": "Logistic Regression",
  "effect_type": "Odds Ratio",
  "effect": 0.6913,
  "confidence_interval": {
    "low": 0.6143,
    "high": 0.7779,
    "level": 0.95
  },
  "p_value": 8.970176340330781e-10,
  "fdr_significant": true,
  "meaningful_effect": true,
  "sample_size": 4024,
  "robustness": {
    "status": "ROBUST",
    "priority_eligible": true,
    "diagnostic_code": null,
    "diagnostic_reason": null
  },
  "controls": [
    "tenure_years",
    "department",
    "role_level",
    "pay_band"
  ],
  "clustered_by": "team_id",
  "health": {
    "analysis_allowed": true,
    "caveats": [
      {
        "code": "NR_LEAVERS",
        "detail": "Survey response rates were lower among employees who later left (57.7%) than among stayers (85.6%). This nonresponse pattern may cause the observed turnover association to be understated.",
        "evidence_strength": "strong",
        "actions": [
          "

In [31]:
# PROCESS 27 – SELECT REQUIRED CAVEATS

required_caveats = (
    q04_llm_payload[
        "health"
    ][
        "caveats"
    ]
)

print(
    json.dumps(
        required_caveats,
        indent=2
    )
)

[
  {
    "code": "NR_LEAVERS",
    "detail": "Survey response rates were lower among employees who later left (57.7%) than among stayers (85.6%). This nonresponse pattern may cause the observed turnover association to be understated.",
    "evidence_strength": "strong",
    "actions": [
      "add_caveat:NR_LEAVERS"
    ]
  }
]


In [32]:
# PROCESS 27A – BUSINESS-FRIENDLY LABELS

CONTROL_LABELS = {
    "tenure_years": "tenure",
    "department": "department",
    "role_level": "role level",
    "pay_band": "pay band",
    "shift_type": "work shift",
    "prior_performance_rating": "prior performance",
    "Q01_below_3_threshold": "very low Q01 scores",
}


def friendly_controls(controls):
    return [
        CONTROL_LABELS.get(c, c)
        for c in controls
    ]


def friendly_caveats(caveats):
    return [
        c["detail"]
        for c in caveats
    ]


print("Business-friendly helper functions ready.")

Business-friendly helper functions ready.


In [33]:
# PROCESS 28 – BUILD Q04 LLM-SAFE INPUT

q04_llm_result = {
    "item": q04_llm_payload["item"],

    "item_label": "Recognition",

    "outcome": q04_llm_payload["outcome"],

    "method": q04_llm_payload["method"],

    "effect_type": q04_llm_payload["effect_type"],

    "effect": q04_llm_payload["effect"],

    "confidence_interval":
        q04_llm_payload["confidence_interval"],

    "sample_size":
        q04_llm_payload["sample_size"],

    "business_metrics":
        q04_llm_payload["business_metrics"],

    "robustness_status":
        q04_llm_payload[
            "robustness"
        ][
            "status"
        ],

    "priority_recommendation_allowed":
        q04_llm_payload[
            "robustness"
        ][
            "priority_eligible"
        ],

    "controls":
        friendly_controls(
            q04_llm_payload["controls"]
        ),

    "required_caveats":
        friendly_caveats(
            q04_llm_payload[
                "health"
            ][
                "caveats"
            ]
        ),

    "calculation_id":
        q04_llm_payload["calculation_id"],
}

print(
    json.dumps(
        q04_llm_result,
        indent=2
    )
)


{
  "item": "Q04",
  "item_label": "Recognition",
  "outcome": "turnover",
  "method": "Logistic Regression",
  "effect_type": "Odds Ratio",
  "effect": 0.6913,
  "confidence_interval": {
    "low": 0.6143,
    "high": 0.7779,
    "level": 0.95
  },
  "sample_size": 4024,
  "business_metrics": {
    "relative_odds_change_pct": -30.9,
    "interpretation_unit": "odds, not probability"
  },
  "robustness_status": "ROBUST",
  "priority_recommendation_allowed": true,
  "controls": [
    "tenure",
    "department",
    "role level",
    "pay band"
  ],
  "required_caveats": [
    "Survey response rates were lower among employees who later left (57.7%) than among stayers (85.6%). This nonresponse pattern may cause the observed turnover association to be understated."
  ],
  "calculation_id": "TURNOVER-Q04-BF5A62B9B6"
}


In [34]:
# PROCESS 28A – BUSINESS-FRIENDLY LABELS

CONTROL_LABELS = {
    "tenure_years": "tenure",
    "department": "department",
    "role_level": "role level",
    "pay_band": "pay band",
    "shift_type": "work shift",
    "prior_performance_rating": "prior performance",
    "Q01_below_3_threshold": "very low Q01 scores",
}


def friendly_controls(controls):
    return [
        CONTROL_LABELS.get(c, c)
        for c in controls
    ]


def friendly_caveats(caveats):
    return [
        c["detail"]
        for c in caveats
    ]

In [35]:
# PROCESS 29 – CONNECT TO GW AZURE OPENAI

import os
from getpass import getpass
from openai import AzureOpenAI

RESOURCE = "gw-sb-10"

ENDPOINT = (
    f"https://{RESOURCE}.openai.azure.com/"
)

api_key = (
    os.getenv("GW_AZURE_OPENAI_KEY")
    or getpass("Azure OpenAI API key: ")
)

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=api_key,
    api_version="2024-12-01-preview",
)

print("Azure OpenAI client ready.")

Azure OpenAI client ready.


In [36]:
# PROCESS 30 – BUILD GROUNDED PROMPT

SYSTEM_PROMPT = """
You are an Employee Engagement Decision Intelligence Agent.

Your role is to explain verified statistical results to
nontechnical HR and business leaders.

STRICT RULES:

 1.⁠ ⁠Use ONLY the numbers and evidence supplied in VERIFIED_RESULT.

 2.⁠ ⁠Never invent, estimate, recalculate, or modify a supplied number.

 3.⁠ ⁠Never claim causation.

 4.⁠ ⁠Use language such as:
   - "associated with"
   - "shows an association"
   - "is linked with"

 5.⁠ ⁠Do NOT use causal terms such as:
   - "causes"
   - "drives"
   - "leads to"

 6.⁠ ⁠Distinguish odds from probability.

 7.⁠ ⁠Include:
   - effect size,
   - 95% confidence interval,
   - sample size.

 8.⁠ ⁠Include all required caveats supplied in VERIFIED_RESULT.

 9.⁠ ⁠Respect the robustness status.

10.⁠ ⁠Do not make decisions or recommendations about individual employees.

11.⁠ ⁠Keep the answer concise and understandable to a nontechnical manager.

12.⁠ ⁠Do not expose p-values unless specifically asked.

13.⁠ ⁠The statistical calculations have already been completed by
    deterministic Python tools. You are only interpreting them.

14.⁠ ⁠When describing response-rate differences, explicitly say
    "survey response rate". Do not describe response rates as
    population shares.

15.⁠ ⁠Do not provide an action recommendation unless the user explicitly
    asks for recommendations or next steps.

16.⁠ ⁠For an Odds Ratio below 1, use the supplied business metric
    to describe the percentage reduction in odds.
    Never describe an odds reduction as a percentage-point reduction
    in probability.

17.⁠ ⁠When appropriate, describe the unit of change explicitly,
    for example:
    "a one-point higher Q12 item score".

18.⁠ ⁠If robustness status is CAUTION:
    - explicitly explain why the finding is uncertain,
    - include the supplied diagnostic reason,
    - do not recommend it as a priority,
    - do not treat statistical significance as sufficient evidence.

19.⁠ ⁠If priority_eligible is false, explicitly state that the finding should not be used as a priority recommendation based on the
    current evidence.

20.⁠ ⁠Display the Calculation ID exactly once, at the end of the response.

21.⁠ ⁠Do not expose internal field names, diagnostic codes, or health-check
    codes unless the user explicitly asks for technical details.
    Translate them into plain business language.

22.⁠ ⁠Do not display boolean implementation fields such as:
    - priority_eligible = true
    - fdr_significant = true
    - meaningful_effect = true
    Instead, express their meaning naturally in business language.

23.⁠ ⁠Translate model control variable names into readable labels.
    For example:
    - tenure_years -> tenure
    - role_level -> role level
    - pay_band -> pay band
    - shift_type -> work shift
    - prior_performance_rating -> prior performance
    - Q01_below_3_threshold -> very low Q01 scores

24.⁠ ⁠Present implementation metadata in natural business language. Never show Python field names, snake_case variable names, internal health-check codes, or internal diagnostic codes.

25.⁠ ⁠When the user asks whether a CAUTION finding should be prioritized,
    lead with the decision conclusion.
    For example:
    "Q07 is associated with absenteeism, but it should not currently
    be prioritized based on the available evidence."
    Do not begin with a standalone "Yes" or "No" that could be ambiguous.

26.⁠ ⁠Never use implementation wording such as:
    - "eligible"
    - "not eligible"
    - "priority_eligible"
    Instead say naturally:
    - "can be considered for prioritization"
    - "should not currently be prioritized"

27.⁠ ⁠Do not repeat the same statistic or sample size more than once unless repetition is necessary for clarity.
"""

In [37]:
# PROCESS 31 – TEST USER QUESTION

q04_user_question = """
What does Q04 Recognition tell us about voluntary turnover?
Explain the finding and its business relevance for an HR Leader
and Business Leader.
"""

In [38]:
# PROCESS 32 – GENERATE Q04 GROUNDED EXPLANATION

verified_result_json = json.dumps(
    q04_llm_result,
    indent=2
)

response_q04 = client.chat.completions.create(
    model="gpt-5-nano",

    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
USER QUESTION:
{q04_user_question}

VERIFIED_RESULT:
{verified_result_json}

Answer the user's question using only the verified result.
"""
        }
    ],

    max_completion_tokens=3000,
)

answer_q04 = (
    response_q04
    .choices[0]
    .message
    .content
    or ""
)

print(answer_q04)

print(
    "\nFinish reason:",
    response_q04.choices[0].finish_reason
)

Q04 Recognition and turnover: what it tells us

- Finding: Recognition (Q04) is associated with voluntary turnover. In a logistic regression that controls for tenure, department, role level, and pay band, the odds ratio is 0.6913 (95% CI: 0.6143 to 0.7779). This is based on a sample of 4,024 employees.
- Interpretation: A one-point higher Recognition score is associated with about a 31% lower odds of turnover (odds ratio of 0.6913). The confidence interval suggests the association could correspond to roughly a 22% to 39% lower odds. This result is an association, not a causal finding.
- Robustness: The result is labeled ROBUST.

Business relevance

- HR Leader: Higher recognition scores are linked with lower odds of voluntary turnover after accounting for key workforce characteristics. This suggests recognition programs or practices may be important to consider as part of retention efforts, though this is an association and not proof of causation.
- Business Leader: The association imp

In [39]:
# PROCESS 33 – TEST A CAUTION FINDING

q07_absence_row = final_findings[
    (final_findings["item"] == "Q07")
    & (final_findings["outcome"] == "absence")
].iloc[0]

q07_absence_payload = build_finding_payload(
    q07_absence_row
)

q07_absence_payload = add_business_metrics(
    q07_absence_payload
)

print(
    json.dumps(
        q07_absence_payload,
        indent=2
    )
)

{
  "dataset": "gallup_v2_5K",
  "item": "Q07",
  "outcome": "absence",
  "method": "Negative Binomial Regression",
  "effect_type": "Incidence Rate Ratio",
  "effect": 0.8124,
  "confidence_interval": {
    "low": 0.7828,
    "high": 0.8431,
    "level": 0.95
  },
  "p_value": 5.497937113130814e-28,
  "fdr_significant": true,
  "meaningful_effect": true,
  "sample_size": 4024,
  "robustness": {
    "status": "CAUTION",
    "priority_eligible": false,
    "diagnostic_code": "X3_TEAM_CONFOUNDING",
    "diagnostic_reason": "Effect shrank by 93.8% within teams; team ICC=0.104."
  },
  "controls": [
    "tenure_years",
    "department",
    "shift_type",
    "Q01_below_3_threshold"
  ],
  "clustered_by": "team_id",
  "health": {
    "analysis_allowed": true,
    "caveats": [
      {
        "code": "NR_SMD_unplanned_absence_days_6m",
        "detail": "Employees who responded to the survey differed in absenteeism from those who did not respond, so nonresponse may affect the estimated assoc

In [40]:
# PROCESS 34 – BUILD Q07 ABSENCE LLM-SAFE INPUT

q07_llm_result = {
    "item":
        q07_absence_payload["item"],

    "item_label":
        "Opinions Count",

    "outcome":
        q07_absence_payload["outcome"],

    "method":
        q07_absence_payload["method"],

    "effect_type":
        q07_absence_payload["effect_type"],

    "effect":
        q07_absence_payload["effect"],

    "confidence_interval":
        q07_absence_payload[
            "confidence_interval"
        ],

    "sample_size":
        q07_absence_payload[
            "sample_size"
        ],

    "business_metrics":
        q07_absence_payload[
            "business_metrics"
        ],

    "robustness_status":
        q07_absence_payload[
            "robustness"
        ][
            "status"
        ],

    "priority_recommendation_allowed":
        q07_absence_payload[
            "robustness"
        ][
            "priority_eligible"
        ],

    "robustness_reason":
        (
            "The estimated association decreased by 93.8% "
            "when employees within the same teams were compared. "
            "This suggests that team-level factors may explain "
            "much of the apparent relationship."
        ),

    "controls":
        friendly_controls(
            q07_absence_payload["controls"]
        ),

    "required_caveats":
        friendly_caveats(
            q07_absence_payload[
                "health"
            ][
                "caveats"
            ]
        ),

    "calculation_id":
        q07_absence_payload[
            "calculation_id"
        ],
}

print(
    json.dumps(
        q07_llm_result,
        indent=2
    )
)

{
  "item": "Q07",
  "item_label": "Opinions Count",
  "outcome": "absence",
  "method": "Negative Binomial Regression",
  "effect_type": "Incidence Rate Ratio",
  "effect": 0.8124,
  "confidence_interval": {
    "low": 0.7828,
    "high": 0.8431,
    "level": 0.95
  },
  "sample_size": 4024,
  "business_metrics": {
    "relative_rate_change_pct": -18.8
  },
  "robustness_status": "CAUTION",
  "priority_recommendation_allowed": false,
  "robustness_reason": "The estimated association decreased by 93.8% when employees within the same teams were compared. This suggests that team-level factors may explain much of the apparent relationship.",
  "controls": [
    "tenure",
    "department",
    "work shift",
    "very low Q01 scores"
  ],
  "required_caveats": [
    "Employees who responded to the survey differed in absenteeism from those who did not respond, so nonresponse may affect the estimated association.",
    "The item shows different response behavior across departments, so departm

In [41]:
# PROCESS 35 – CAUTION TEST QUESTION

q07_user_question = """
Is Q07 Opinions Count an important engagement factor associated
with absenteeism?

Explain the finding, its limitations, and whether it should be
prioritized for an HR Leader and Business Leader.
"""

In [42]:
# PROCESS 36 – GENERATE Q07 GROUNDED EXPLANATION

verified_result_json = json.dumps(
    q07_llm_result,
    indent=2
)

response_q07 = client.chat.completions.create(
    model="gpt-5-nano",

    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
USER QUESTION:
{q07_user_question}

VERIFIED_RESULT:
{verified_result_json}

Answer the user's question using only the verified result.
"""
        }
    ],

    max_completion_tokens=3000,
)

answer_q07 = (
    response_q07
    .choices[0]
    .message
    .content
    or ""
)

print(answer_q07)

print(
    "\nFinish reason:",
    response_q07.choices[0].finish_reason
)

Q07 (Opinions Count) is associated with absenteeism. In a negative binomial model, a one-point higher Q07 score is linked with a lower rate of absence:

- Incidence Rate Ratio (IRR): 0.8124
- 95% Confidence Interval: 0.7828 to 0.8431
- Relative rate change: -18.8%
- Sample size: 4,024
- Model controls: tenure, department, work shift, and very low Q01 scores

Important caveats and limitations:
- Robustness status is CAUTION. The estimated association decreased by 93.8% when comparing employees within the same teams, suggesting team-level factors may explain much of the apparent relationship.
- Required caveats: 
  - Employees who responded to the survey differed in absenteeism from those who did not respond, so nonresponse may affect the estimated association.
  - The item shows different response behavior across departments, so department context should be considered when interpreting the result.

Priority for action:
- This finding should not currently be prioritized based on the avai

In [43]:
# PROCESS 37 – DEBUG LLM RESPONSES

print("Q04")
print(
    "Finish reason:",
    response_q04.choices[0].finish_reason
)
print("Usage:")
print(response_q04.usage)

print("\n" + "=" * 60 + "\n")

print("Q07")
print(
    "Finish reason:",
    response_q07.choices[0].finish_reason
)
print("Usage:")
print(response_q07.usage)

Q04
Finish reason: stop
Usage:
CompletionUsage(completion_tokens=2431, prompt_tokens=1177, total_tokens=3608, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=2112, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 28, 'engine_ttlt_ms': 10516, 'pre_inference_ms': 225, 'service_tbt_ms': 4, 'service_ttft_ms': 485, 'service_ttlt_ms': 10918, 'user_visible_ttft_ms': 260})


Q07
Finish reason: stop
Usage:
CompletionUsage(completion_tokens=2680, prompt_tokens=1230, total_tokens=3910, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=2432, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 19, 'engine_ttlt_ms': 11679, 'pre_inference_ms': 105, '